# Data Extraction Module

auth: Rudy Alvarez Martin (617-869-4992) 

This script includes :

    ✅ Smart chunking for academic text (sentence-based with sub-sentence handling)
    ✅ Page continuity validation (detects and corrects split sentences across pages)
    ✅ One JSON per page structure (ensuring lightweight processing and scalability)
    ✅ Optional page merging (appends first sentence of the next page if needed)


In [ ]:
import json
import io
import os
import logging
import boto3
import re
import nltk
from datetime import datetime
from PyPDF2 import PdfReader
from typing import List, Dict

# Initialize S3 client
s3 = boto3.client('s3')

# Define Configuration Dictionary
CONFIG = {
    "bucket_name": "sagemaker-us-east-1-188494237500",
    "pdf_folder": "dev/pdf/arxiv_wellsfargo",
    "pages_folder": "dev/page",
    "json_folder": "dev/json",
    "chunk_size": 200,  # Max words per chunk
    "pages_per_file": 1,
    "pdf_password": None
}

# Configure Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Ensure NLTK dependencies are available
nltk.download('punkt')

def smart_chunking(text, max_words=200):
    """Splits academic text into semantically meaningful chunks using sentence + sub-sentence splitting."""
    sentences = nltk.sent_tokenize(text)  # Split text into sentences
    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence_length = len(sentence.split())

        # If sentence is too long, attempt secondary splitting at logical points
        if sentence_length > max_words:
            sub_sentences = re.split(r'[;,:]|\b(and|but|or|which|that)\b', sentence)  # Break on logical points
            sub_sentences = [s.strip() for s in sub_sentences if s]  # Remove empty splits
        else:
            sub_sentences = [sentence]

        for sub_sentence in sub_sentences:
            sub_length = len(sub_sentence.split())

            # Start a new chunk if adding this sub-sentence exceeds max_words
            if current_length + sub_length > max_words:
                chunks.append(" ".join(current_chunk))
                current_chunk = []
                current_length = 0

            current_chunk.append(sub_sentence)
            current_length += sub_length

    # Add the last chunk if it contains any sentences
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

def validate_page_continuity(text_current: str, text_next: str) -> bool:
    """Checks if the current page ends mid-sentence and suggests merging the next page's first sentence."""
    sentences = nltk.sent_tokenize(text_current)

    # If the last sentence does not end with '.', '!', or '?', assume it's incomplete
    if sentences and not sentences[-1].endswith(('.', '!', '?')):
        logging.warning("⚠️ Possible broken sentence at page boundary. Consider appending the next page's first sentence.")
        return True  # Suggest merging with the next page

    return False  # No continuity issues detected

def chunk_text_into_segments(pdf_page_data: Dict, max_words: int = None) -> List[Dict]:
    """Splits academic text into semantically meaningful chunks before embedding."""
    try:
        max_words = max_words or CONFIG["chunk_size"]
        text = pdf_page_data.get('text', '')

        if not text.strip():
            return []

        page_number = pdf_page_data.get("page_number", 1)
        document_name = pdf_page_data.get("document_name", "unknown.pdf").replace(".pdf", "")

        chunks = []
        chunked_texts = smart_chunking(text, max_words)  # Use smart chunking

        for i, chunk_text in enumerate(chunked_texts):
            chunk_id = f"{document_name}_{page_number}_{i+1:03d}"
            chunks.append({"chunk_id": chunk_id, "text": chunk_text, "embedding": None})

        return chunks

    except Exception as e:
        logging.error(f"❌ Error chunking text: {e}")
        return []

def save_page_chunks_as_json_s3(original_key: str, chunks: List[Dict], bucket: str = None):
    """Saves all text chunks from a single page into one JSON file."""
    try:
        bucket = bucket or CONFIG["bucket_name"]
        json_folder = CONFIG["json_folder"]

        document_name = os.path.basename(original_key)
        page_number = original_key.split("_")[-1].split(".")[0]
        text_date = datetime.now().isoformat()

        chunk_data = {
            "document_name": document_name,
            "page_number": page_number,
            "text_date": text_date,
            "chunks": chunks
        }

        chunk_key = f"{json_folder}/page_{page_number}.json"
        s3.put_object(Bucket=bucket, Key=chunk_key, Body=json.dumps(chunk_data).encode('utf-8'))

        logging.info(f"✅ Saved all chunks for page {page_number} in {chunk_key}")

    except Exception as e:
        logging.error(f"❌ Error saving page chunks to JSON: {e}")

def extract_text_from_pdf_page_s3(page_key: str, next_page_key: str = None, bucket: str = None) -> Dict:
    """Extracts text from a single-page PDF and optionally appends the first sentence of the next page."""
    try:
        bucket = bucket or CONFIG["bucket_name"]
        response = s3.get_object(Bucket=bucket, Key=page_key)
        pdf_file = response["Body"].read()

        reader = PdfReader(io.BytesIO(pdf_file))
        text = reader.pages[0].extract_text() or ""
        page_number = int(page_key.split("_")[-1].split(".")[0])

        # Try to capture page continuity if a next page exists
        if next_page_key:
            try:
                next_response = s3.get_object(Bucket=bucket, Key=next_page_key)
                next_pdf_file = next_response["Body"].read()
                next_reader = PdfReader(io.BytesIO(next_pdf_file))
                next_text = next_reader.pages[0].extract_text() or ""

                # Validate if current page ends mid-sentence
                if validate_page_continuity(text, next_text):
                    next_sentences = nltk.sent_tokenize(next_text)
                    if next_sentences:
                        text += " " + next_sentences[0]  # Append first sentence from the next page

            except Exception as e:
                logging.warning(f"⚠️ Could not retrieve next page {next_page_key}: {e}")

        return {"document_name": os.path.basename(page_key), "page_number": page_number, "text": text.strip()}

    except Exception as e:
        logging.error(f"❌ Error extracting text from {page_key}: {e}")
        return {"page_number": None, "text": ""}

def process_documents():
    """Processes all PDFs, extracts pages, text, and saves JSON chunks."""
    try:
        pdf_keys = list_objects_s3()
        logging.info(f"📄 Processing {len(pdf_keys)} PDFs...")

        for pdf_key in pdf_keys:
            logging.info(f"🔍 Processing PDF: {pdf_key}")
            split_pdf_into_pages_s3(pdf_key)

            page_keys = list_objects_s3(prefix=CONFIG["pages_folder"])

            for i, page_key in enumerate(page_keys):
                next_page_key = page_keys[i+1] if i+1 < len(page_keys) else None
                pdf_page_data = extract_text_from_pdf_page_s3(page_key, next_page_key)
                chunks = chunk_text_into_segments(pdf_page_data)

                if chunks:
                    save_page_chunks_as_json_s3(page_key, chunks)

            logging.info(f"✅ Finished processing PDF: {pdf_key}")

    except Exception as e:
        logging.error(f"❌ Error processing documents: {e}")
